# Контроль отображения R1.4

Четыре статуса ниже — **синтетический тест цветов**, не результаты научной карты. Отдельно проверяется кнопка при повреждённом архиве.

In [ ]:
from pathlib import Path
import sys, json, tempfile
root=Path.cwd()
while root != root.parent and not (root/'efc/model.py').exists(): root=root.parent
sys.path.insert(0,str(root))
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib','inline')
import matplotlib.pyplot as plt
from IPython.display import display
from efc.map_view import STYLES, configuration_map_figure
from efc.dashboard import Dashboard
control=dict(verification='ALL_DISPLAY_CHAINS_CHECKED', rows=[
    dict(n_fog=30,buffer=b,arrival_rate_s=100.,status=s,display_chain_checked=True)
    for b,s in zip((0,2,5,10),STYLES)])

In [ ]:
fig=configuration_map_figure(control,100.)
fig.axes[0].set_title('Синтетический контроль цветов — НЕ результаты модели')
assert len(set(fig.axes[0].images[0].cmap.colors))==4
assert 'Недостаточно\nданных' in [t.get_text() for t in fig.axes[0].texts]
display(fig)
plt.close(fig)

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    Path(folder,'map.json').write_text('{broken')
    panel=Dashboard(folder,render=False)
    panel.button.click()
    assert panel.last_error is None
    assert panel.last_saved_error is not None
    assert panel.last['saved']['status']=='ARCHIVE_UNVERIFIED'
    assert panel.last['result']['metrics']['arrivals']==600
    assert panel.last['result']['display_summary']['intervals'] is None
    assert not panel.last['saved']['rows']
    assert not panel.button.disabled
print(json.dumps(dict(archive_error_isolated=True,figure_states_distinct=True,
    fixture_is_scientific_result=False,forecast_executed=False,full_grid_executed=False)))